In [1]:
import pandas as pd
import os

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

meta_df_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df_ea["sample_id"] = meta_df_ea["sample_id"].astype(str)
meta_df_ea = meta_df_ea.set_index("sample_id")

print("EA full cohort:", len(meta_df_ea))
print("\nCPD stats:")
print(meta_df_ea['cpd'].describe())

smoker_ids_ea = meta_df_ea[meta_df_ea["smoking_status"] == "Smoker"].index.tolist()
print("\nEA smokers:", len(smoker_ids_ea))

print("\nAny smokers with cpd <= 0?")
print(meta_df_ea[(meta_df_ea['smoking_status']=='Smoker') & (meta_df_ea['cpd']<=0)].shape[0])

# Check EA's CPD SNP list and confounders (from your original file listing)
df_cpd_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint9_doubleml_stability_results_cpd.csv"))
print("\nEA CPD SNP stability file shape:", df_cpd_ea.shape)
print((df_cpd_ea['stability_fraction']==0).sum(), "of", len(df_cpd_ea), "have stability_fraction==0")

import numpy as np
confounders_cpd_ea = np.load(os.path.join(ea_dir, "confounders_X_cpd.npy"))
print("\nEA CPD confounders shape:", confounders_cpd_ea.shape, "(should match", len(smoker_ids_ea), "smokers)")

EA full cohort: 1460

CPD stats:
count    1452.000000
mean       15.200413
std        15.013954
min         0.000000
25%         0.000000
50%        20.000000
75%        30.000000
max        65.000000
Name: cpd, dtype: float64

EA smokers: 795

Any smokers with cpd <= 0?
0

EA CPD SNP stability file shape: (37013, 3)
36997 of 37013 have stability_fraction==0

EA CPD confounders shape: (793, 12) (should match 795 smokers)


In [2]:
print("Samples with null CPD:", meta_df_ea['cpd'].isna().sum())
print(meta_df_ea[meta_df_ea['cpd'].isna()]['smoking_status'].value_counts())

Samples with null CPD: 8
smoking_status
Non-smoker    6
Smoker        2
Name: count, dtype: int64


In [3]:
print("EA CPD SNP file check:")
print(df_cpd_ea['stability_fraction'].describe())

# Check if any smokers have missing CPD (would explain the 793 vs 795 gap)
smokers_with_valid_cpd = meta_df_ea[(meta_df_ea['smoking_status']=='Smoker') & (meta_df_ea['cpd'].notna())]
print("\nSmokers with valid (non-null) CPD:", len(smokers_with_valid_cpd))

EA CPD SNP file check:
count    37013.000000
mean         0.000134
std          0.009984
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: stability_fraction, dtype: float64

Smokers with valid (non-null) CPD: 793


In [4]:
import numpy as np

smoker_cols_ea_cpd = meta_df_ea[(meta_df_ea['smoking_status']=='Smoker') & (meta_df_ea['cpd'].notna())].index.tolist()

# Must also intersect with actual genotype matrix columns
encoded_df_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
sample_cols_all_ea = encoded_df_ea.columns[1:].tolist()
smoker_cols_ea_cpd = [c for c in smoker_cols_ea_cpd if c in sample_cols_all_ea]
print("EA CPD-eligible smokers found in genotype matrix:", len(smoker_cols_ea_cpd))

pheno_cpd_ea = meta_df_ea.loc[smoker_cols_ea_cpd, "cpd"].astype(float)
print("EA CPD phenotype aligned:", pheno_cpd_ea.shape)

# Order verification: rebuild PCA from this exact sample order, compare to saved confounders
from sklearn.decomposition import PCA

smoker_geno_ea = encoded_df_ea.set_index("probe_id")[smoker_cols_ea_cpd].T
smoker_geno_ea_std = (smoker_geno_ea - smoker_geno_ea.mean(axis=0)) / smoker_geno_ea.std(axis=0)
smoker_geno_ea_std = smoker_geno_ea_std.fillna(0)

pca_ea = PCA(n_components=5, svd_solver='full')
pcs_rebuilt_ea = pca_ea.fit_transform(smoker_geno_ea_std.values)

saved_pc1_ea = confounders_cpd_ea[:, 0]
rebuilt_pc1_ea = pcs_rebuilt_ea[:, 0]

corr_ea_order = np.corrcoef(saved_pc1_ea, rebuilt_pc1_ea)[0, 1]
print(f"\nOrder verification correlation: {corr_ea_order:.4f}")

EA CPD-eligible smokers found in genotype matrix: 793
EA CPD phenotype aligned: (793,)

Order verification correlation: 0.1174


In [5]:
# Try natural file order: filter meta_df_ea by criteria but WITHOUT reordering via .loc[list]
meta_df_ea_reset = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df_ea_reset["sample_id"] = meta_df_ea_reset["sample_id"].astype(str)

smoker_natural_order = meta_df_ea_reset[
    (meta_df_ea_reset['smoking_status']=='Smoker') & (meta_df_ea_reset['cpd'].notna())
]
smoker_natural_order = smoker_natural_order[smoker_natural_order['sample_id'].isin(sample_cols_all_ea)]
print("Natural order count:", len(smoker_natural_order))

smoker_cols_natural = smoker_natural_order['sample_id'].tolist()
print("Same set as before?", set(smoker_cols_natural) == set(smoker_cols_ea_cpd))
print("Same order as before?", smoker_cols_natural == smoker_cols_ea_cpd)

Natural order count: 793
Same set as before? True
Same order as before? True


In [6]:
# Check all 12 columns against rebuilt PC1-5, look for ANY strong match
for col_idx in range(12):
    saved_col = confounders_cpd_ea[:, col_idx]
    for pc_idx in range(5):
        rebuilt_col = pcs_rebuilt_ea[:, pc_idx]
        corr = np.corrcoef(saved_col, rebuilt_col)[0, 1]
        if abs(corr) > 0.5:
            print(f"Saved col {col_idx} vs rebuilt PC{pc_idx+1}: r = {corr:.4f}")

print("\nIf nothing printed above, no column matches any rebuilt PC well.")

# Also check basic stats of each saved column, to see if any look like age/gender rather than PCA scores
print("\nSaved confounder column stats:")
for col_idx in range(12):
    col = confounders_cpd_ea[:, col_idx]
    print(f"Col {col_idx}: mean={col.mean():.3f}, std={col.std():.3f}, min={col.min():.3f}, max={col.max():.3f}")


If nothing printed above, no column matches any rebuilt PC well.

Saved confounder column stats:
Col 0: mean=0.000, std=43.158, min=-21.078, max=187.310
Col 1: mean=0.000, std=25.779, min=-33.744, max=89.618
Col 2: mean=-0.000, std=15.103, min=-25.020, max=61.078
Col 3: mean=-0.000, std=12.791, min=-25.193, max=55.628
Col 4: mean=-0.000, std=11.446, min=-36.841, max=31.023
Col 5: mean=-0.000, std=10.775, min=-27.073, max=36.577
Col 6: mean=-0.000, std=10.325, min=-42.875, max=28.664
Col 7: mean=-0.000, std=10.237, min=-23.246, max=37.142
Col 8: mean=0.000, std=9.992, min=-70.910, max=28.207
Col 9: mean=0.000, std=9.899, min=-31.596, max=82.757
Col 10: mean=-0.000, std=0.999, min=-2.021, max=3.001
Col 11: mean=0.501, std=0.500, min=0.000, max=1.000


In [7]:
# Full SNP matrix for these exact 793 samples (not just genic CPD subset)
full_geno_ea_cpd_samples = encoded_df_ea.set_index("probe_id")[smoker_cols_ea_cpd].T
full_geno_std = (full_geno_ea_cpd_samples - full_geno_ea_cpd_samples.mean(axis=0)) / full_geno_ea_cpd_samples.std(axis=0)
full_geno_std = full_geno_std.fillna(0)

pca_full = PCA(n_components=5, svd_solver='full')
pcs_full_rebuild = pca_full.fit_transform(full_geno_std.values)

for col_idx in [0,1,2]:  # just check first 3 saved PCs against rebuild
    for pc_idx in range(5):
        corr = np.corrcoef(confounders_cpd_ea[:, col_idx], pcs_full_rebuild[:, pc_idx])[0, 1]
        if abs(corr) > 0.5:
            print(f"Saved col {col_idx} vs full-SNP rebuilt PC{pc_idx+1}: r = {corr:.4f}")

print("Done checking.")

Done checking.


In [8]:
# Rebuild EA CPD confounders cleanly: PCA (full SNP set, svd_solver='full') + age + gender
# for exactly these 793 samples, in this exact order

age_ea = meta_df_ea.loc[smoker_cols_ea_cpd, "age"].astype(float).values
gender_ea = meta_df_ea.loc[smoker_cols_ea_cpd, "gender"].values
gender_ea_bin = (gender_ea == "Male").astype(float)  # confirm this encoding matches your convention

pca_10 = PCA(n_components=10, svd_solver='full')
pcs_10 = pca_10.fit_transform(full_geno_std.values)  # reuse full_geno_std from previous cell

age_std = (age_ea - age_ea.mean()) / age_ea.std()

confounders_cpd_ea_rebuilt = np.column_stack([pcs_10, age_std, gender_ea_bin])
print("Rebuilt EA CPD confounders shape:", confounders_cpd_ea_rebuilt.shape)

np.save(os.path.join(ea_dir, "confounders_X_cpd_EA_rebuilt.npy"), confounders_cpd_ea_rebuilt)
print("Saved.")

Rebuilt EA CPD confounders shape: (793, 12)
Saved.


In [9]:
import re
import json
import gc

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

# EA CPD SNP list -> gene mapping
snp_list_cpd_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint9_doubleml_stability_results_cpd.csv"))
snp_list_cpd_ea["core_name"] = snp_list_cpd_ea["probe_id"].map(strip_address_suffix)
snp_list_cpd_ea = snp_list_cpd_ea.merge(pos_lookup, left_on="core_name", right_index=True, how="left")
snp_list_cpd_ea = snp_list_cpd_ea.dropna(subset=["Chr", "MapInfo"])
print("EA CPD SNPs with resolved positions:", len(snp_list_cpd_ea))

genes_clean = pd.read_csv(os.path.join(r"C:\Users\user\Downloads\GSE148375_clean", "grch37_genes_clean.csv"))
protein_coding_genes = set(genes_clean[genes_clean["Gene type"] == "protein_coding"]["gene_name"])

snp_to_gene_cpd_ea = {}
for chrom in snp_list_cpd_ea["Chr"].unique():
    chrom_str = str(chrom)
    genes_this_chrom = genes_clean[genes_clean["chrom"] == chrom_str].sort_values("start")
    snps_this_chrom = snp_list_cpd_ea[snp_list_cpd_ea["Chr"].astype(str) == chrom_str]
    if len(genes_this_chrom) == 0 or len(snps_this_chrom) == 0:
        continue
    starts = genes_this_chrom["start"].values
    ends = genes_this_chrom["end"].values
    names = genes_this_chrom["gene_name"].values
    for _, row in snps_this_chrom.iterrows():
        pos = row["MapInfo"]
        idx = np.searchsorted(starts, pos, side="right") - 1
        match = None
        for j in range(max(0, idx-3), min(len(starts), idx+4)):
            if starts[j] <= pos <= ends[j]:
                match = names[j]
                break
        snp_to_gene_cpd_ea[row["probe_id"]] = match if match else "intergenic"

print("EA CPD SNPs mapped:", len(snp_to_gene_cpd_ea))
with open(os.path.join(ea_dir, "snp_to_gene_map_cpd_EA.json"), "w") as f:
    json.dump(snp_to_gene_cpd_ea, f)

# Build signed gene burden matrix
snp_gene_series_cpd_ea = pd.Series(snp_to_gene_cpd_ea)
snp_gene_series_cpd_ea = snp_gene_series_cpd_ea[snp_gene_series_cpd_ea != "intergenic"]

encoded_genic_cpd_ea = encoded_df_ea[encoded_df_ea["probe_id"].isin(snp_gene_series_cpd_ea.index)].copy()
encoded_genic_cpd_ea["gene"] = encoded_genic_cpd_ea["probe_id"].map(snp_gene_series_cpd_ea)
encoded_genic_cpd_ea = encoded_genic_cpd_ea[["probe_id", "gene"] + smoker_cols_ea_cpd]
print("EA CPD genic SNPs:", len(encoded_genic_cpd_ea))

geno_matrix_cpd_ea = encoded_genic_cpd_ea[smoker_cols_ea_cpd].values
pheno_vals_cpd_ea = pheno_cpd_ea.values

geno_centered = geno_matrix_cpd_ea - geno_matrix_cpd_ea.mean(axis=1, keepdims=True)
pheno_centered = pheno_vals_cpd_ea - pheno_vals_cpd_ea.mean()
numerator = geno_centered @ pheno_centered
denom = np.sqrt((geno_centered**2).sum(axis=1) * (pheno_centered**2).sum())
denom[denom == 0] = np.nan
corr = numerator / denom
direction = np.sign(np.nan_to_num(corr, nan=0.0))
direction[direction == 0] = 1
print("Flipped:", (direction < 0).sum(), "| Kept:", (direction >= 0).sum())

flipped = np.where(direction[:, None] < 0, 2 - geno_matrix_cpd_ea, geno_matrix_cpd_ea)
encoded_genic_cpd_ea[smoker_cols_ea_cpd] = flipped

gene_burden_cpd_ea = encoded_genic_cpd_ea.groupby("gene")[smoker_cols_ea_cpd].sum()
gene_burden_cpd_ea_pc = gene_burden_cpd_ea[gene_burden_cpd_ea.index.isin(protein_coding_genes)]
print("EA CPD gene burden (protein-coding):", gene_burden_cpd_ea_pc.shape)

gene_burden_cpd_ea_pc.to_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_CPD_EA.csv"))
print("Saved.")

EA CPD SNPs with resolved positions: 37013
EA CPD SNPs mapped: 37013
EA CPD genic SNPs: 29675
Flipped: 15486 | Kept: 14189
EA CPD gene burden (protein-coding): (10326, 793)
Saved.


In [10]:
# Verify rebuilt confounders' PCA columns actually match the same-order full-SNP PCA we already validated
corr_check = np.corrcoef(confounders_cpd_ea_rebuilt[:, 0], pcs_full_rebuild[:, 0])[0, 1]
print(f"Rebuilt confounders col 0 vs pcs_full_rebuild PC1: r = {corr_check:.4f}")
print("(expect ~1.0, since confounders_cpd_ea_rebuilt was built directly from this same PCA)")

# Sanity check age/gender columns landed correctly
print("\nAge column (col 10) stats:", confounders_cpd_ea_rebuilt[:,10].mean(), confounders_cpd_ea_rebuilt[:,10].std())
print("Gender column (col 11) unique values:", np.unique(confounders_cpd_ea_rebuilt[:,11]))

Rebuilt confounders col 0 vs pcs_full_rebuild PC1: r = 1.0000
(expect ~1.0, since confounders_cpd_ea_rebuilt was built directly from this same PCA)

Age column (col 10) stats: -2.4416506367544425e-16 1.0
Gender column (col 11) unique values: [0. 1.]
